# Curvature Estimation — IMU + GPS (EKF Fusion)

Computes two curvature types from ZED IMU and GPS MCAP recordings:

| Output | Formula | Meaning |
|--------|---------|----------|
| `κ_terrain_spline` | Frenet curvature on spline-fit elevation | How curved the road profile is (up/down) |
| `κ_terrain_pitch` | `dα/ds` (pitch rate along path) | Faster terrain curvature estimate |
| `κ_path` | `ω_z / v` | Horizontal path curvature (turning radius) |

Speed `v` is obtained by fusing ZED pose velocity and GPS velocity through a 1-D Kalman filter.

## 1. Config & Imports
Change file paths here.

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import scipy.interpolate as sc
from pyproj import Transformer
from filterpy.kalman import KalmanFilter
import os

IMU_FILE_PATH = r"D:\Data_gathered\2026_05_22\Camera\10_50_00\back_22_05_2026-10_50_00.mcap"
GPS_FILE_PATH = r"D:\Data_gathered\2026_05_22\Rosbag\10_50_00\rosbag\rosbag_0.mcap"

TIME       = 1779439917.972930459  # reference Unix timestamp (same as other validation files)
DELTA_M    = 0.1                   # sampling interval for profile window (m)
WIN_BEFORE = 5.0                   # metres before reference point to include
WIN_AFTER  = 20.0                  # metres after reference point to include
MIN_SPEED  = 0.5                   # m/s — below this κ_path is set to NaN
SMOOTH_W   = 21                    # Savitzky-Golay window for pitch smoothing (set to 0 to disable)
GPS_WINDOW = 15.0                  # seconds either side of TIME to read from GPS MCAP

## 2. Load IMU from MCAP

- `/zed/pose` → position (x, y, z) + orientation quaternion → pitch
- `/zed/imu/data` → angular velocity z (yaw rate ω_z)

In [ ]:
POSE_TOPIC = "/zed/pose"
IMU_TOPIC  = "/zed/imu/data"

pose_data = {"t": [], "x": [], "y": [], "z": [], "qx": [], "qy": [], "qz": [], "qw": []}
imu_raw   = {"t": [], "omega_z": []}

print("Reading ZED MCAP...")
with open(IMU_FILE_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[POSE_TOPIC, IMU_TOPIC]):
        if channel.topic == POSE_TOPIC:
            t = ros_msg.header.stamp.sec + ros_msg.header.stamp.nanosec / 1e9
            pose_data["t"].append(t)
            pose_data["x"].append(ros_msg.pose.position.x)
            pose_data["y"].append(ros_msg.pose.position.y)
            pose_data["z"].append(ros_msg.pose.position.z)
            pose_data["qx"].append(ros_msg.pose.orientation.x)
            pose_data["qy"].append(ros_msg.pose.orientation.y)
            pose_data["qz"].append(ros_msg.pose.orientation.z)
            pose_data["qw"].append(ros_msg.pose.orientation.w)
        elif channel.topic == IMU_TOPIC:
            t = ros_msg.header.stamp.sec + ros_msg.header.stamp.nanosec / 1e9
            imu_raw["t"].append(t)
            imu_raw["omega_z"].append(ros_msg.angular_velocity.z)

print(f"Pose samples : {len(pose_data['t'])}")
print(f"IMU  samples : {len(imu_raw['t'])}")

## 3. Load GPS from MCAP

In [ ]:
GPS_TOPIC = "/navsat_topic"

gps_raw = {"t": [], "lat": [], "lon": [], "alt": []}

# Only decode the 2*GPS_WINDOW second slice around TIME — much faster on large rosbags
gps_start_ns = int((TIME - GPS_WINDOW) * 1_000_000_000)
gps_end_ns   = int((TIME + GPS_WINDOW) * 1_000_000_000)

print(f"Reading GPS MCAP ({2*GPS_WINDOW:.0f}s window around TIME)...")
with open(GPS_FILE_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(
        topics=[GPS_TOPIC], start_time=gps_start_ns, end_time=gps_end_ns
    ):
        gps_raw["t"].append(message.publish_time / 1e9)
        gps_raw["lat"].append(ros_msg.latitude)
        gps_raw["lon"].append(ros_msg.longitude)
        gps_raw["alt"].append(ros_msg.altitude)

print(f"GPS samples : {len(gps_raw['t'])}")

# Convert to Dutch RD New (EPSG:28992)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
rd_xs, rd_ys = transformer.transform(gps_raw["lon"], gps_raw["lat"])
rd_xs  = np.array(rd_xs)
rd_ys  = np.array(rd_ys)
gps_t  = np.array(gps_raw["t"])

# Cumulative arc-length along GPS track
gps_s = np.concatenate([[0.0], np.cumsum(np.hypot(np.diff(rd_xs), np.diff(rd_ys)))])
print(f"GPS track length: {gps_s[-1]:.1f} m")

## 4. EKF Speed Fusion (ZED pose + GPS)

A 1-D Kalman filter fuses scalar speed from ZED pose differentiation and GPS track differentiation.
- ZED speed is high-frequency and smooth but in a local frame.
- GPS speed is lower-frequency but absolute.

Speed magnitude is frame-independent, so no coordinate alignment is needed.

In [ ]:
pose_t  = np.array(pose_data["t"])
pose_x  = np.array(pose_data["x"])
pose_y  = np.array(pose_data["y"])

# ZED scalar speed: |Δpos| / Δt
pose_dt    = np.diff(pose_t)
pose_dist  = np.hypot(np.diff(pose_x), np.diff(pose_y))
v_zed_raw  = np.where(pose_dt > 1e-6, pose_dist / pose_dt, 0.0)
v_zed_t    = 0.5 * (pose_t[:-1] + pose_t[1:])          # midpoint timestamps

# GPS scalar speed: |Δdist| / Δt
gps_dt     = np.diff(gps_t)
gps_dist   = np.diff(gps_s)
v_gps_raw  = np.where(gps_dt > 1e-6, gps_dist / gps_dt, 0.0)
v_gps_t    = 0.5 * (gps_t[:-1] + gps_t[1:])

# ── 1-D Kalman filter: state = [v, a] ────────────────────────────────────────
# Run on the ZED time axis (higher rate). GPS is interpolated as a second measurement.
kf = KalmanFilter(dim_x=2, dim_z=1)

dt0 = float(np.median(pose_dt))

kf.x = np.array([[float(v_zed_raw[0])], [0.0]])   # [v, a]
kf.P = np.diag([4.0, 1.0])                         # initial covariance
kf.H = np.array([[1.0, 0.0]])                       # observe speed only
kf.R = np.array([[0.25]])                           # ZED measurement noise variance (m/s)²

sigma_j = 0.5   # jerk noise (m/s³) — tune if needed
kf.Q = np.array([
    [dt0**4 / 4 * sigma_j**2,  dt0**3 / 2 * sigma_j**2],
    [dt0**3 / 2 * sigma_j**2,  dt0**2     * sigma_j**2],
])

v_fused = np.zeros(len(v_zed_t))
R_gps   = np.array([[1.0]])  # GPS measurement noise variance (m/s)² — higher than ZED

for i in range(len(v_zed_t)):
    dt = float(pose_dt[i]) if pose_dt[i] > 1e-6 else dt0
    kf.F = np.array([[1.0, dt], [0.0, 1.0]])
    kf.predict()

    # Primary update: ZED speed
    kf.R = R_gps if False else np.array([[0.25]])   # ZED noise
    kf.update(np.array([[v_zed_raw[i]]]))

    # Secondary update: GPS speed when a GPS sample is close in time
    idx_gps = np.searchsorted(v_gps_t, v_zed_t[i])
    if 0 < idx_gps < len(v_gps_t):
        dt_gps = abs(v_gps_t[idx_gps] - v_zed_t[i])
        if dt_gps < 0.12:   # within ~half a GPS epoch
            kf.R = R_gps
            kf.update(np.array([[v_gps_raw[idx_gps]]]))

    v_fused[i] = max(kf.x[0, 0], 0.0)   # speed is non-negative

# Cumulative arc-length on ZED pose (for distance axis)
pose_s = np.concatenate([[0.0], np.cumsum(pose_dist)])
v_fused_at_pose = np.interp(pose_t, v_zed_t, v_fused)

print(f"EKF done. Mean fused speed: {v_fused.mean():.2f} m/s, max: {v_fused.max():.2f} m/s")

## 5. Elevation Profile

Pitch is extracted from the ZED pose orientation quaternion and integrated over distance.

In [ ]:
def quaternion_to_pitch(qx, qy, qz, qw):
    qx, qy, qz, qw = np.array(qx), np.array(qy), np.array(qz), np.array(qw)
    sinp = 2 * (qw * qy - qz * qx)
    sinp = np.clip(sinp, -1, 1)
    return np.degrees(np.arcsin(sinp))

pitch_raw  = quaternion_to_pitch(pose_data["qx"], pose_data["qy"], pose_data["qz"], pose_data["qw"])
pitch_bias = np.mean(pitch_raw)
print(f"Pitch bias: {pitch_bias:.2f} deg")
pitch = pitch_raw - pitch_bias

if SMOOTH_W > 0:
    pitch = savgol_filter(pitch, window_length=SMOOTH_W, polyorder=3)

# ds along ZED pose path
ds = np.concatenate([[0.0], pose_dist])

# Elevation by integrating pitch over distance
imu_elevation = np.cumsum(ds * np.sin(np.radians(pitch)))

# Window around reference time
ref_idx = np.argmin(np.abs(pose_t - TIME))
dt_err  = abs(pose_t[ref_idx] - TIME)
print(f"Reference TIME={TIME}, closest pose Δt={dt_err:.3f}s (index {ref_idx})")

s_rel  = pose_s - pose_s[ref_idx]
mask   = (s_rel >= -WIN_BEFORE) & (s_rel <= WIN_AFTER)

s_win      = s_rel[mask]
z_win      = imu_elevation[mask] - imu_elevation[mask][0]
pitch_win  = pitch[mask]
pitch_rad_win = np.radians(pitch_win)

print(f"Window: {s_win[0]:.1f} m to {s_win[-1]:.1f} m, {mask.sum()} samples")

## 6a. Vertical Terrain Curvature

**Method A** — Frenet curvature on spline fit (same as `validation_main.py`):
$$\kappa = \frac{|z''|}{(1 + z'^2)^{3/2}}$$

**Method B** — Direct pitch differentiation:
$$\kappa \approx \frac{d\alpha}{ds}$$
Valid for small pitch angles where $\cos(\alpha) \approx 1$.

In [ ]:
# Method A: spline
sp_z    = sc.make_splrep(s_win, z_win, s=0)
dzds    = sp_z.derivative(nu=1)(s_win)
d2zds2  = sp_z.derivative(nu=2)(s_win)
kappa_terrain_spline = np.abs(d2zds2) / (1.0 + dzds**2)**1.5

# Method B: direct pitch differentiation
kappa_terrain_pitch = np.gradient(pitch_rad_win, s_win)

print(f"κ_terrain spline  — max: {np.nanmax(kappa_terrain_spline):.4f} 1/m")
print(f"κ_terrain pitch   — max: {np.nanmax(np.abs(kappa_terrain_pitch)):.4f} 1/m")

## 6b. Horizontal Path Curvature

$$\kappa_{\text{path}} = \frac{\omega_z}{v}$$

- $\omega_z$: yaw rate from `/zed/imu/data` (rad/s)
- $v$: EKF-fused scalar ground speed (m/s)
- Set to NaN when $v < $ `MIN_SPEED` to avoid noise at low speed.

In [ ]:
# Interpolate ω_z and fused speed onto the window's pose timestamps
omega_z_all = np.interp(pose_t, np.array(imu_raw["t"]), np.array(imu_raw["omega_z"]))
omega_z_win = omega_z_all[mask]
speed_win   = v_fused_at_pose[mask]

kappa_path = np.where(speed_win > MIN_SPEED, omega_z_win / speed_win, np.nan)

n_nan = int(np.sum(np.isnan(kappa_path)))
print(f"κ_path — max: {np.nanmax(np.abs(kappa_path)):.4f} 1/m")
print(f"  NaN (speed < {MIN_SPEED} m/s): {n_nan} / {len(kappa_path)} samples")

## 7. Plots

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12))
ax_v, ax_z, ax_kv = axes
"""
# Speed
ax_v.plot(v_zed_t - v_zed_t[0], v_zed_raw,  label="ZED (raw)",   color="steelblue",  alpha=0.5, linewidth=1)
ax_v.plot(v_gps_t - v_gps_t[0], v_gps_raw,  label="GPS (raw)",   color="orange",    alpha=0.5, linewidth=1)
ax_v.plot(v_zed_t - v_zed_t[0], v_fused,    label="EKF fused",   color="black",     linewidth=2)
ax_v.axhline(MIN_SPEED, color="red", linestyle="--", linewidth=1, label=f"MIN_SPEED={MIN_SPEED} m/s")
ax_v.set_xlabel("Time (s)"); ax_v.set_ylabel("Speed (m/s)")
ax_v.set_title("EKF Speed Fusion (ZED + GPS)")
ax_v.legend(); ax_v.grid(True)
"""
# Elevation profile
ax_z.plot(s_win, z_win, label="IMU elevation (pitch integration)", color="blue")
ax_z.set_xlabel("Distance from reference (m)"); ax_z.set_ylabel("Elevation (m, relative)")
ax_z.set_title("Elevation Profile")
ax_z.legend(); ax_z.grid(True)

# Vertical terrain curvature
ax_kv.plot(s_win, kappa_terrain_spline,            label="Spline method",         color="orange")
ax_kv.plot(s_win, kappa_path, label="ω_z / v  (EKF speed)", color="red")
ax_kv.plot(s_win, np.abs(kappa_terrain_pitch),     label="Pitch diff. method",    color="purple", linestyle="--")
ax_kv.set_xlabel("Distance (m)"); ax_kv.set_ylabel("κ (1/m)")
ax_kv.set_title("Vertical Terrain Curvature")
ax_kv.axhline(0, color="k", linewidth=0.8, linestyle="--")
ax_kv.legend(); ax_kv.grid(True)
plt.tight_layout()
plt.show()

## 8. Save Output

Saves a `.npz` compatible with `validation_main.py`.

In [ ]:
gps_parts   = GPS_FILE_PATH.replace("/", os.sep).split(os.sep)
date_folder = gps_parts[-5]
time_folder = gps_parts[-3]

SAVE_DIR = os.path.join(r"D:\Validation_results", date_folder, time_folder)
os.makedirs(SAVE_DIR, exist_ok=True)

method = "EKF_curvature_validation"
print(z_win)
np.savez_compressed(
    os.path.join(SAVE_DIR, f"{method}_{TIME}.npz"),
    x      = None,
    y      = None,
    z      = z_win,
    s      = s_win,
    t      = [TIME],
    method = np.array([method]),
    kappa_terrain_spline = kappa_terrain_spline,
    kappa_terrain_pitch  = kappa_terrain_pitch,
    kappa_path           = kappa_path,
    gps_lon =   gps_raw["lon"],
    gps_lat =   gps_raw["lat"],
)
fpath = os.path.join(SAVE_DIR, f"{method}_{TIME}.npz")
print(f"Saved to {fpath}")
print(f"  Core   : x, y, z, s, t, method")
print(f"  Extras : kappa_terrain_spline, kappa_terrain_pitch, kappa_path")

method = "Z_positional_tracking"

x = np.array(pose_data["x"])
y = np.array(pose_data["y"])
z = np.array(pose_data["z"])
t = np.array(pose_data["t"])

ref_idx = np.argmin(np.abs(t - TIME)) #argmin returns the index of the minimum value in the array, which corresponds to the closest timestamp to TIME
dt = abs(pose_data["t"][ref_idx] - TIME)
print(f"Using TIME = {TIME}, closest GPS message Δt = {dt:.3f}s " f"(index {ref_idx})")


s = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
cum_s = np.concatenate([[0], np.cumsum(s)])
cum_s_from_s0 = cum_s - cum_s[ref_idx]
idx_25m = np.argmax(cum_s_from_s0 >= 20.0)
idx_5m = np.argmin(cum_s_from_s0 <= -5.0)
mask = (t >= t[idx_5m]) & (t <= t[idx_25m])
s_rel = cum_s_from_s0[mask]
z_rel = z[mask] - z[ref_idx]
t_rel = t[mask]

np.savez_compressed(
    os.path.join(SAVE_DIR, f"{method}_{TIME}.npz"),
    x      = None,
    y      = None,
    z      = z_rel,
    s      = s_rel,
    t      = [TIME],
    method = np.array([method]),
)
fpath = os.path.join(SAVE_DIR, f"{method}_{TIME}.npz")
print(f"Saved to {fpath}")
print(f"  Core   : x, y, z, s, t, method")
